In [ ]:
#    MARS SEASONS
#    ~~~~~~~~~~~~
#
#    - Mars mission data processor by season.
#    - Author: Raul Jesus Lopez @la9una
#    - GitHub: https://github.com/la9una/mars_habitability
#
#    This script processes environmental data from the Mars2020 mission,
#    adding a 'Season' column based on Martian solar longitude (L_s) 
#    and generating a representative sample for each season.
#    Only data from Sol 63 onward are included, following MEDA guidelines.
#    
#    Actions:
#    1. Download data from GitHub.
#    2. Filter data to start from Sol 63 (see: https://atmos.nmsu.edu/PDS/data/PDS4/Mars2020/mars2020_meda/readme.txt).
#    3. Add seasons based on L_s:
#       - Spring: 0° ≤ L_s < 90°
#       - Summer: 90° ≤ L_s < 180°
#       - Autumn: 180° ≤ L_s < 270°
#       - Winter: 270° ≤ L_s < 360°
#    4. Create a stratified sample:
#       - Select 30 samples or 10% of records per season (whichever is larger).
#       - Include all records if fewer than 30 are available.
#    5. Save:
#       - Full dataset with seasons, sorted by Season and Sol.
#       - Representative sample.
#
#    Data Source:
#    - Time equivalence table from Perseverance (Mars 2020) Analyst's Notebook:
#      https://an.rsl.wustl.edu/m20/AN/an3.aspx
#
#    Output Files:
#    - generated_files/generated_mars2020_time_table_with_seasons.csv: Full dataset with seasons.
#    - generated_files/generated_mars2020_time_table_sample.csv: Representative sample.

import pandas as pd
import os
import glob
import re
from ipywidgets import interact, Dropdown
from IPython.display import display
try:
    from google.colab import files
except ImportError:
    files = None

# Cargar el archivo de tiempo
archivo_tiempo = pd.read_csv('mars2020_time_table_sorted_by_season_and_sol.csv')

# Definir las variables físicas y sus rutas correspondientes
variables_fisicas = {
    'ATS': 'physical_vars/CAL_ATS/',
    'RDS': 'physical_vars/CAL_RDS/',
    'PS': 'physical_vars/DER_PS/',
    'RHS': 'physical_vars/DER_RHS/',
    'TIRS': 'physical_vars/DER_TIRS/'
}

# Crear un menú desplegable interactivo
variable_dropdown = Dropdown(
    options=list(variables_fisicas.keys()),
    description='Variable:',
    disabled=False,
)

def procesar_variable(variable_seleccionada):
    ruta_variable = variables_fisicas[variable_seleccionada]
    
    # Verificar si la carpeta de la variable seleccionada existe
    if not os.path.isdir(ruta_variable):
        print(f"La carpeta para la variable '{variable_seleccionada}' no está presente.")
        return

    # Inicializar una lista para almacenar los datos promedio de cada sol para la variable seleccionada
    variable_data = []

    # Expresión regular para extraer el número de sol de los nombres de archivo
    patron_sol = re.compile(r"WE__(\d{4})")  # Busca el número de cuatro dígitos después de 'WE__'

    # Leer cada archivo de la variable en la carpeta
    for archivo in glob.glob(os.path.join(ruta_variable, "*.csv")):
        try:
            # Extraer el número de sol del nombre del archivo
            match = patron_sol.search(os.path.basename(archivo))
            if match:
                sol_numero = int(match.group(1))
            else:
                print(f"No se pudo extraer el número de sol del archivo '{archivo}'. Saltando...")
                continue

            # Leer el archivo de la variable
            variable_df = pd.read_csv(archivo)
            variable_df['Sol'] = sol_numero  # Agregar columna de sol

            # Seleccionar las columnas de la variable específica para calcular el promedio
            if variable_seleccionada == 'ATS':
                columnas_variable = ['ATS_LOCAL_TEMP1', 'ATS_LOCAL_TEMP2', 'ATS_LOCAL_TEMP3', 'ATS_LOCAL_TEMP4', 'ATS_LOCAL_TEMP5']
                variable_df[f'{variable_seleccionada}_PROMEDIO'] = variable_df[columnas_variable].mean(axis=1)

            elif variable_seleccionada == 'RDS':
                columnas_lat = [col for col in variable_df.columns if col.startswith('RDS_LAT_')]
                columnas_top = [col for col in variable_df.columns if col.startswith('RDS_TOP_')]
                variable_df['RDS_LAT_PROMEDIO'] = variable_df[columnas_lat].mean(axis=1)
                variable_df['RDS_TOP_PROMEDIO'] = variable_df[columnas_top].mean(axis=1)
                variable_df[f'{variable_seleccionada}_PROMEDIO'] = variable_df[['RDS_LAT_PROMEDIO', 'RDS_TOP_PROMEDIO']].mean(axis=1)

            elif variable_seleccionada == 'PS':
                variable_df = variable_df[variable_df['PRESSURE_MEASUREMENT_MODE'] == 'nominal']
                columnas_variable = ['BAROCAP1_PRESSURE', 'BAROCAP2_PRESSURE', 'BAROCAP3_PRESSURE']
                variable_df[f'{variable_seleccionada}_PROMEDIO'] = variable_df[columnas_variable].mean(axis=1)

            elif variable_seleccionada == 'RHS':
                columnas_variable = ['HUMIDITY_LOCAL_TEMP']
                variable_df[f'{variable_seleccionada}_PROMEDIO'] = variable_df[columnas_variable].mean(axis=1)

            elif variable_seleccionada == 'TIRS':
                columnas_variable = ['DOWNWARD_LW_IRRADIANCE', 'UPWARD_LW_IRRADIANCE']
                variable_df[f'{variable_seleccionada}_PROMEDIO'] = variable_df[columnas_variable].mean(axis=1)

            # Obtener el promedio por sol y agregarlo a la lista
            promedio_sol = variable_df.groupby('Sol')[f'{variable_seleccionada}_PROMEDIO'].mean().reset_index()
            variable_data.append(promedio_sol)

        except Exception as e:
            print(f"Error al procesar el archivo '{archivo}': {e}")

    # Combinar todos los DataFrames de la variable en uno solo
    if variable_data:
        variable_combinado = pd.concat(variable_data, ignore_index=True)

        # Unir el archivo de tiempo para asociar cada sol con su estación
        datos_combinados = pd.merge(archivo_tiempo[['Sol', 'Season']], variable_combinado, on='Sol', how='inner')

        # Calcular el promedio por estación para la variable seleccionada
        promedio_por_estacion = datos_combinados.groupby('Season')[f'{variable_seleccionada}_PROMEDIO'].mean().reset_index()

        # Mostrar el DataFrame resultante
        display(promedio_por_estacion)

        # Guardar el archivo CSV
        nombre_archivo = f'promedio_{variable_seleccionada}_por_estacion.csv'
        promedio_por_estacion.to_csv(nombre_archivo, index=False)
        print(f"Archivo '{nombre_archivo}' guardado exitosamente.")

        # Descargar en Google Colab si se está ejecutando allí
        if files is not None:
            files.download(nombre_archivo)

    else:
        print(f"No se encontraron datos para la variable '{variable_seleccionada}'.")

# Llamar a la función con el selector interactivo
interact(procesar_variable, variable_seleccionada=variable_dropdown)
